In [105]:
from stanza.utils.conll import CoNLL
import pandas as pd
import os
import Utils
from collections import Counter
import json

# Identifying Conditionals in corrected texts

We use the corrected texts because they contain the target hypotheses and we are not doing any accuracy-related research.

In [106]:
cor_path = f"../parsed_documents/corrected/"
cor_files = sorted(os.listdir(cor_path))

In [107]:
def get_if(sentence):
    for w in sentence.all_words:
        if w.lemma.lower() == "if":
            return w
    return None    

        

def get_conditionals(doc, student_id, group, topic, writing_id):
    """
    Returns a list of dictionaries with information about the conditional clauses found in the Stanza doc.
    The dictionaries include information about the conditional
    """
    dict_list = []
    
    # Each instance of a conditional corresponds to a dict in the return dict_list
    for cor_sent in doc.sentences:
        found_if = get_if(cor_sent)
        
        # No "if" found in the sentence
        if not found_if:
            continue
        
        if_clause_head = cor_sent.all_words[found_if.head - 1]
        
        # Relationship between the head of the sentence and the head of the If-clause: If ccomp, if is a complementizer, not a conditional marker
        if if_clause_head.deprel and "ccomp" in if_clause_head.deprel:
            continue
        
        if_head_children = Utils.get_children_ud(cor_sent, if_clause_head)
        
        had_as_if_clause_aux = len([child for child in if_head_children if "aux" in child.deprel and child.lemma == "have" and "Tense=Past" in child.feats]) != 0
         
        # We're looking for the VERB of the if-clause, not just the semantic head. If the verb is a copula/aux and is in the past, then it should count as a conditional Type II
        for child in if_head_children:
            if child.lemma != "have" and child.xpos == "VBD" and "comp" not in child.deprel: # No "have": prevent 3rd conditional
                if "pass" in child.deprel: 
                    print(f"Child: {child.text}, Head: {if_clause_head.text} Sent: {cor_sent.text}")
                    break
                if_clause_head = child
                if_head_children = Utils.get_children_ud(cor_sent, if_clause_head)
                break 
            
        
        if_clause_aux = [c for c in if_head_children if c.deprel != "aux:pass" and "aux" in c.deprel] # allow only for passive auxiliaries
        main_clause_head = None
        main_clause_modal = None
        aux_main_clause = None
        
        

        for word in cor_sent.all_words:

            if (word.head == 0):
                # Ideally, a verb will be the head, but it may also be a noun or adjective (attributive sentences)
                if "VB" in word.xpos:
                    main_clause_head = word
                
                else:
                    main_clause_head = [c for c in Utils.get_children_ud(cor_sent, word) if "VB" in c.xpos][0]
                    
                aux_main_clause =  [c for c in Utils.get_children_ud(cor_sent, main_clause_head) if "aux" in c.deprel]
            
            else:
                # modals allowed in apodosis of CII
                is_cii_aux = word.lemma.lower() in {"could", "would", "might", "may"}
                # whether the word is inside the if-clause
                is_in_if_clause = word.head == found_if.head
                
                if (is_cii_aux and not is_in_if_clause and main_clause_modal is None):
                    main_clause_modal = word
        

        if_head_past = if_clause_head.feats and "Tense=Past" in if_clause_head.feats
        
        
        
        # Add identified sentence to df 
        cond_dict = {
                    "topic": topic,
                    "student_id": student_id,
                    "writing_id": writing_id,
                    "group": group,
                    "is_cii": not had_as_if_clause_aux and if_head_past and main_clause_modal is not None and not if_clause_aux,
                    "cor_sent" : cor_sent.text,
                    "cor_if_head": if_clause_head.text,
                    "cor_if_head_lemma": if_clause_head.lemma,
                    "cor_main_head": main_clause_head.text,
                    "cor_main_aux": aux_main_clause[0].text if aux_main_clause else None,
                }
               
        dict_list.append(cond_dict)
                        
        
    return dict_list




## Running the code on our data

In [108]:
output_path = "../Python output files/"
os.makedirs(output_path, exist_ok=True)

topic_word_count = Counter()

def update_topic_words_counter(doc, topic):
    n_words = sum(len(sent.all_words) for sent in doc.sentences)
    topic_word_count[topic] += n_words
            

In [109]:
output_file = output_path + "/02_all_conditionals.csv"
log_file = f"{output_path}/unprocessed_filenames.txt"



# Load existing parsed data
if os.path.exists(output_file):
    print("Output file exists: resuming run")
    found_conds = pd.read_csv(output_file)
    cond_list = found_conds.to_dict("records")
    processed_ids = set(found_conds["writing_id"].astype(str))
else:
    cond_list = []
    processed_ids = set()



# Load unprocessed log safely
if os.path.exists(log_file):
    with open(log_file, "r") as f:
        unprocessed_files = f.read().splitlines()
else:
    unprocessed_files = []



# Parse
for i, filename in enumerate(cor_files):

    if i % 1000 == 0:
        print(f"File {i} of {len(cor_files)}")

    filename_split = filename.split("_")

    student_id = filename_split[-1][:-6]
    group = filename_split[0]
    topic = filename_split[1]
    writing_id = filename_split[2]  
    
    path2file = os.path.join(cor_path, filename)

    # Skip if already processed
    if writing_id in processed_ids:
        # print("File already processed")
        continue

    # Skip missing files
    if not os.path.isfile(path2file):
        continue

    # Parse document
    doc = None
    try:
        doc = CoNLL.conll2doc(path2file)
        cond_clauses_dicts = get_conditionals(
            doc, student_id, group, topic, writing_id
        )
        
    except Exception:
        unprocessed_files.append(filename)
        continue
    
    # Count the number of words that this file has added to its "topic"
    update_topic_words_counter(doc, topic)
    
    # Append conditional results
    for d in cond_clauses_dicts:
        cond_list.append(d)

    # Mark as processed
    processed_ids.add(writing_id)

    # Periodic save
    if i % 3000 == 0 and len(cond_list) > 0:
        print("Saving checkpoint...")
        pd.DataFrame(cond_list).to_csv(output_file, index=False)
        print("Saved.")    


# Save final results
found_conds = pd.DataFrame(cond_list)
found_conds.to_csv(output_file, index=False)



cii = found_conds[found_conds["is_cii"] == True]
cii.to_csv(
    output_path + "/02_conditionals_ii.csv", #TODO reset
    index=False
)



File 0 of 33496
Saving checkpoint...
Saved.
Child: was, Head: missed Sent: If the shooting was missed, it is turn to the team who gets the rebound to start their offense from the half-court line.
Child: were, Head: caught Sent: If you were caught by him, you will replace him until you catch one.
File 1000 of 33496
File 2000 of 33496
Child: was, Head: killed Sent: If the kingdom was killed, villains are winning, on the contrary, the kingdom and loyal ministers are winning.
Child: were, Head: killed Sent: If all of your characters were killed, you lose the game.
File 3000 of 33496
Saving checkpoint...
Saved.
Child: were, Head: finished Sent: The first one to finish he has to say bus complete and the others have to stop even if they were not finished and then they collect their points.
Child: were, Head: worked Sent: I have the chance to discover and complete my degrees with an experience that I'd never have if I were worked in another firm.
File 4000 of 33496
Child: was, Head: recognized

In [110]:

with open(f"{output_path}/topic_word_count.json", "w") as count_file:
    json.dump(dict(topic_word_count), count_file)

# Save unprocessed files
existing_unprocessed = set()

if os.path.exists(log_file):
    with open(log_file, "r") as f:
        existing_unprocessed = set(f.read().splitlines())

new_unprocessed = [f for f in unprocessed_files if f not in existing_unprocessed]

with open(log_file, "a") as f:
    for fname in new_unprocessed:
        f.write(fname + "\n")

In [111]:
len(cii)

897